# Homework 3

In [1]:
import pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()


df1 = pd.read_csv("C:/Users/crist/Downloads/PL-season-2324.csv")
df2 = pd.read_csv("C:/Users/crist/Downloads/PL-season-2425.csv")
df1.head()
df2.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,16/08/24,Man United,Fulham,1,0,H,0,0,D,R Jones,...,5,2,12,10,7,8,2,3,0,0
1,17/08/24,Ipswich,Liverpool,0,2,A,0,0,D,T Robinson,...,2,5,9,18,2,10,3,1,0,0
2,17/08/24,Arsenal,Wolves,2,0,H,1,0,H,J Gillett,...,6,3,17,14,8,2,2,2,0,0
3,17/08/24,Everton,Brighton,0,3,A,0,1,A,S Hooper,...,1,5,8,8,1,5,1,1,1,0
4,17/08/24,Newcastle,Southampton,1,0,H,1,0,H,C Pawson,...,1,4,15,16,3,12,2,4,1,0


In [2]:
df1["Season"] = "2023-24"
df2["Season"] = "2024-25"
df = pd.concat([df1, df2], ignore_index=True)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["GD"] = df["FTHG"] - df["FTAG"]

def points(row):
    if row["FTR"] == "H":
        return 3
    elif row["FTR"] == "A":
        return 0
    else:
        return 1

df["HomePoints"] = df.apply(lambda r: points(r), axis=1)
df["AwayPoints"] = df.apply(lambda r: 3 - points(r) if r["FTR"] != "D" else 1, axis=1)
df.head()


C:\Users\crist\AppData\Local\Temp\ipykernel_23108\578446023.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HC,AC,HY,AY,HR,AR,Season,GD,HomePoints,AwayPoints
0,2023-11-08,Burnley,Man City,0,3,A,0,2,A,C Pawson,...,6,5,0,0,1,0,2023-24,-3,0,3
1,2023-12-08,Arsenal,Nott'm Forest,2,1,H,2,0,H,M Oliver,...,8,3,2,2,0,0,2023-24,1,3,0
2,2023-12-08,Bournemouth,West Ham,1,1,D,0,0,D,P Bankes,...,10,4,1,4,0,0,2023-24,0,1,1
3,2023-12-08,Brighton,Luton,4,1,H,1,0,H,D Coote,...,6,7,2,2,0,0,2023-24,3,3,0
4,2023-12-08,Everton,Fulham,0,1,A,0,0,D,S Attwell,...,10,4,0,2,0,0,2023-24,-1,0,3


The count of Points will be useful for questions regarding team effectiveness and seeing the results of matches. The points function is based on EPL scoring, If home gets 3 (home win), away gets 0, If home gets 0 (away win), away gets 3, If they draw, both get 1 → handled by the else 1. 

In [3]:
home = df[[
    "Season","Date","HomeTeam","FTHG","FTAG","GD","HomePoints","HS","HST","HC"
]].rename(columns={
    "HomeTeam":"Team",
    "FTHG":"GoalsFor",
    "FTAG":"GoalsAgainst",
    "HomePoints":"Points",
    "HS":"Shots",
    "HST":"ShotsOnTarget",
    "HC":"Corners"
})

away = df[[
    "Season","Date","AwayTeam","FTAG","FTHG","GD","AwayPoints","AS","AST","AC"
]].rename(columns={
    "AwayTeam":"Team",
    "FTAG":"GoalsFor",
    "FTHG":"GoalsAgainst",
    "AwayPoints":"Points",
    "AS":"Shots",
    "AST":"ShotsOnTarget",
    "AC":"Corners"
})

team_df = pd.concat([home, away], ignore_index=True)
team_df.head()

,Season,Date,Team,GoalsFor,GoalsAgainst,GD,Points,Shots,ShotsOnTarget,Corners
0,2023-24,2023-11-08,Burnley,0,3,-3,0,6,1,6
1,2023-24,2023-12-08,Arsenal,2,1,1,3,15,7,8
2,2023-24,2023-12-08,Bournemouth,1,1,0,1,14,5,10
3,2023-24,2023-12-08,Brighton,4,1,3,3,27,12,6
4,2023-24,2023-12-08,Everton,0,1,-1,0,19,9,10


Here I'm renaming columns for ease of understanding, keeoping only relevant information, and putting the two teams in the same column for ease of use in visualizations.

In [4]:
team_df = team_df.sort_values(["Season","Team","Date"])
team_df["Matchweek"] = team_df.groupby(["Season","Team"]).cumcount() + 1

team_df.head()

,Season,Date,Team,GoalsFor,GoalsAgainst,GD,Points,Shots,ShotsOnTarget,Corners,Matchweek
130,2023-24,2023-02-12,Arsenal,2,1,1,3,19,6,4,1
38,2023-24,2023-03-09,Arsenal,3,1,2,3,17,5,12,2
866,2023-24,2023-04-11,Arsenal,0,1,1,0,14,1,11,3
901,2023-24,2023-05-12,Arsenal,4,3,-1,3,23,9,8,4
79,2023-24,2023-08-10,Arsenal,1,0,1,3,12,2,5,5


This is to determine matchweek, which is asked about in Question 2. 

In [5]:
team_df["GoalsFor_roll"] = (
    team_df.groupby(["Season","Team"])["GoalsFor"]
           .transform(lambda s: s.rolling(3, min_periods=1).mean())
)

team_df["Shots_roll"] = (
    team_df.groupby(["Season","Team"])["Shots"]
           .transform(lambda s: s.rolling(3, min_periods=1).mean())
)

team_df["ShotsOnTarget_roll"] = (
    team_df.groupby(["Season","Team"])["ShotsOnTarget"]
           .transform(lambda s: s.rolling(3, min_periods=1).mean())
)

team_df["Corners_roll"] = (
    team_df.groupby(["Season","Team"])["Corners"]
           .transform(lambda s: s.rolling(3, min_periods=1).mean())
)

team_df.head()

,Season,Date,Team,GoalsFor,GoalsAgainst,GD,Points,Shots,ShotsOnTarget,Corners,Matchweek,GoalsFor_roll,Shots_roll,ShotsOnTarget_roll,Corners_roll
130,2023-24,2023-02-12,Arsenal,2,1,1,3,19,6,4,1,2.000000,19.000000,6.0,4.000000
38,2023-24,2023-03-09,Arsenal,3,1,2,3,17,5,12,2,2.500000,18.000000,5.5,8.000000
866,2023-24,2023-04-11,Arsenal,0,1,1,0,14,1,11,3,1.666667,16.666667,4.0,9.000000
901,2023-24,2023-05-12,Arsenal,4,3,-1,3,23,9,8,4,2.333333,18.000000,5.0,10.333333
79,2023-24,2023-08-10,Arsenal,1,0,1,3,12,2,5,5,1.666667,16.333333,4.0,8.000000


Again, preparing for Question 2 that recommends using rolling averages for data visualization.

### Question 1

In [6]:
team_select = alt.selection_point(fields=["Team"], empty="none")
season_select = alt.selection_point(fields=["Season"], bind=alt.binding_select(options=["2023-24","2024-25"]))
brush_extremes = alt.selection_interval()

metric_select = alt.selection_point(
    fields=["metric"],
    bind=alt.binding_select(options=["GoalsFor_roll","Shots_roll","ShotsOnTarget_roll","Corners_roll"]),
    value="GoalsFor_roll"
)

team_summary = team_df.groupby(["Season","Team"], as_index=False).agg({
    "Points":"sum",
    "GoalsFor":"sum",
    "GoalsAgainst":"sum",
    "GD":"sum"
})

q1 = (alt.Chart(team_summary).mark_point(size=140).encode(
        x=alt.X("Points:Q", title="Total Points"),
        y=alt.Y("Team:N", sort="-x"),
        color=alt.Color("Team:N", legend=alt.Legend(title="Team", orient="right")),
        shape=alt.Shape(
            "Season:N",
            scale=alt.Scale(
                domain=["2023-24", "2024-25"],
                range=["circle", "triangle-up"]),
            legend=alt.Legend(
                title="Season (Shape Encoding)",
                orient="left",
                symbolSize=200,
                symbolStrokeWidth=2)
        ),
        tooltip=["Team","Season","Points","GD"],
        opacity=alt.condition(season_select, alt.value(1), alt.value(0.2))
    )
    .add_params(team_select, season_select)
    .properties(title="Q1: Team Performance Across Seasons (Click to Select Team, Dropdown to Select Season)")
)

For Question 1, I used the recommended dot plot because it is a simple and legible way to encode the total points while still allowing all teams to appear in one graph  The biggest design decision involved how to visually distinguish seasons while still preserving team identity. Since team color is essential in the graph, I couldn’t use color to encode season. Instead, I introduced shape encoding (circle for 2023–24, triangle for 2024–25) and forced Altair to render a separate shape legend. This ensures that season differences are readable at a glance without interfering with color (used for teams) or opacity (used for season filter). Interactivity plays a supporting role here, as you can select a  single team's highlights across all filtered views. Without interactivity, the chart would still be interpretable, but the ability to isolate a team and follow it through the dashboard significantly strengthens the analytical experience.

### Question 2 

In [7]:
melted = team_df.melt(
    id_vars=["Season","Team","Matchweek"],
    value_vars=["GoalsFor_roll","Shots_roll","ShotsOnTarget_roll","Corners_roll"],
    var_name="metric",
    value_name="value"
)

hover_team = alt.selection_point(
    fields=["Team"],
    on="mouseover",
    empty="none"
)

q2 = (
    alt.Chart(melted)
    .mark_line()
    .encode(
        x=alt.X("Matchweek:Q"),
        y=alt.Y("value:Q", title="Rolling Average"),
        color=alt.Color("Team:N"),
        opacity=alt.condition(
            hover_team,
            alt.value(1),
            alt.value(0.15)
        ),
        strokeWidth=alt.condition(
            hover_team,
            alt.value(3),
            alt.value(1)
        ),
        tooltip=["Team","Season","Matchweek","value"]
    )
    .transform_filter(season_select)
    .transform_filter(metric_select)
    .add_params(metric_select, hover_team)
    .properties(title="Q2: Attacking Performance Over Time (Hover to Highlight)")
)

The second question required showing how a team’s attacking performance evolves over time, which suggests a line chart with matchweek on the x‑axis. However, plotting all teams simultaneously produces a dense plot that obscures trends. The key design decision was to introduce hover‑based highlighting, which allows the viewer to isolate a single team’s trajectory without removing the contextual presence of others. The metric toggle (seprated by goals, shots, shots on target, and corners) reflects the assignment’s emphasis on dynamic querying. Instead of hard‑coding one metric, the user can switch between multiple attacking indicators, making the chart a flexible and interactive analytical tool. Without interactivity, this view would require multiple separate charts or small multiples, which would dilute the narrative coherence of the dashboard.

### Question 4

In [8]:
q4_scatter = (
    alt.Chart(df)
    .mark_circle(size=80)
    .encode(
        x=alt.X("FTHG:Q", title="Home Goals"),
        y=alt.Y("FTAG:Q", title="Away Goals"),
        color=alt.condition(brush_extremes, alt.value("red"), alt.value("steelblue")),
        tooltip=["Date","HomeTeam","AwayTeam","FTHG","FTAG","FTR"]
    )
    .add_params(brush_extremes)
    .properties(title="Q4: Extreme Match Outcomes (Brush to Select)")
)

q4_table = (
    alt.Chart(df)
    .transform_filter(brush_extremes)
    .transform_calculate(
        MatchLabel='datum.HomeTeam + " " + datum.FTHG + "–" + datum.FTAG + " " + datum.AwayTeam'
    )
    .transform_window(
        row_number="row_number()"
    )
    .transform_filter("datum.row_number <= 15")
    .mark_text(align="left")
    .encode(
        y=alt.Y("row_number:O", axis=None),
        text="MatchLabel:N"
    )
    .properties(title="Selected Matches (Top 15)")
)

For Question 4, the goal was to help users identify and explore matches with unusually high or low goal totals. A scatter plot of home goals vs. away goals is a natural fit and was recommended in the document. High‑scoring matches cluster in the upper‑right, while low‑scoring matches sit near the axes. The central design choice was to use brushing to select extreme regions. Brushing is intuitive for this task because extremity is spatial: users can simply drag a rectangle around the area of interest. I redesigned the table to show full match labels (e.g., “Arsenal 3–1 Chelsea”) and ensured that the row numbering occurs after filtering so the “Top 15” list is accurate and readable. This view would lose most of its analytical value without interactivity. The scatter plot alone does show the distribution of outcomes, but brushing transforms it into a discovery tool for users, who can now isolate outliers and compare clusters.

## Final Dashboard

In [9]:
dashboard = alt.vconcat(
    q1,
    q2,
    alt.hconcat(q4_scatter, q4_table)
).resolve_scale(color="independent")

dashboard

alt.VConcatChart(...)